# Práctica 3: Machine Learning — Integrante 3 Jorge (K-NN + Naïve Bayes)

| Día | Actividad | Secciones |
|---|---|---|
| 3 de noviembre | Preparación de bibliotecas y entorno de trabajo | 1 |
| 4 de noviembre | Datasets, análisis exploratorio y normalización | 2, 3, 4 y 5 |
| 5 de noviembre | K-NN desde cero | 6 |
| 6 de noviembre | Naïve Bayes con Scikit-learn | 7 |

> Para usarlo: **Entorno de ejecución → Ejecutar todas**. Cada vez que abras el notebook, vuelve a ejecutarlo
> completo, porque Colab borra las variables al desconectarse.

# 1. Preparación del entorno

En esta sección se configura el entorno de trabajo utilizado para desarrollar la práctica de clasificación mediante
algoritmos de Machine Learning.

Se emplea Google Colab como plataforma de ejecución y Python como lenguaje de programación. Asimismo, se importan las
bibliotecas necesarias para la generación, manipulación, visualización y evaluación de los conjuntos de datos, así
como para la implementación de los clasificadores.

Con el propósito de obtener resultados reproducibles y facilitar la comparación entre los integrantes del equipo, se
utilizan configuraciones comunes de experimentación: semilla aleatoria, proporción de entrenamiento y prueba,
número de folds y funciones de evaluación.

## 1.1 Importación de bibliotecas

- **NumPy:** operaciones matemáticas y manipulación de arreglos numéricos.
- **Pandas:** organización y exploración de datos mediante estructuras tabulares.
- **Matplotlib y Seaborn:** gráficas para visualizar la distribución de los datos y los resultados.
- **Time:** medición de los tiempos de ejecución de los algoritmos.
- **Scikit-learn:** generación de datasets, particiones de validación, normalización, métricas y los clasificadores
  que se implementan con biblioteca (Naïve Bayes).

Google Colab ya incluye todas estas bibliotecas, por lo que no es necesario instalarlas con `pip`.

In [ ]:
import time
import copy
import platform
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

from sklearn.datasets import make_classification, load_wine
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier   # solo para VALIDAR el K-NN propio
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore", category=FutureWarning)
print("Entorno preparado correctamente")

## 1.2 Configuración común del equipo

Estos parámetros son los mismos que usa el Integrante 1 en su notebook (`random_state=42`, `test_size=0.30`,
`StratifiedKFold` con 5 folds). Los tres integrantes deben usarlos sin cambios para que las particiones sean
idénticas y los resultados de los seis clasificadores puedan compararse.

In [ ]:
SEMILLA   = 42     # semilla común para todo el equipo
TEST_SIZE = 0.30   # Hold Out 70/30
N_FOLDS   = 5      # K-Fold Cross-Validation (k = 5)

np.random.seed(SEMILLA)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

print(f"Python       {platform.python_version()}")
for nombre, mod in [("numpy", np), ("pandas", pd), ("scikit-learn", sklearn),
                    ("matplotlib", plt.matplotlib), ("seaborn", sns)]:
    print(f"{nombre:<12} {mod.__version__}")

## 1.3 Archivo `requirements.txt`

Se genera un archivo con las versiones exactas de las bibliotecas de este entorno. Se sube al repositorio de GitHub
para que cualquier persona pueda reproducir los resultados.

In [ ]:
paquetes = {"numpy": np, "pandas": pd, "scikit-learn": sklearn,
            "matplotlib": plt.matplotlib, "seaborn": sns}
with open("requirements.txt", "w") as f:
    for nombre, mod in paquetes.items():
        f.write(f"{nombre}=={mod.__version__}\n")
print(open("requirements.txt").read())

## 1.4 Funciones de evaluación compartidas

Para que los seis clasificadores se midan de la misma forma, el equipo utiliza estas funciones:

- **Hold Out 70/30 estratificado**: conserva la proporción de clases en entrenamiento y prueba.
- **K-Fold estratificado** con `shuffle=True` y la semilla común.
- La **normalización se ajusta solo con los datos de entrenamiento** de cada partición, lo que evita
  *data leakage*.
- Se mide por separado el tiempo de entrenamiento (`fit`) y el de predicción (`predict`).
- Métricas: exactitud, precisión, recall y F1, con promedio *macro* porque hay tres clases.

Funcionan con cualquier modelo que tenga los métodos `fit(X, y)` y `predict(X)`: los clasificadores programados
desde cero (Distancia Mínima, 1-NN y K-NN) y los de Scikit-learn (Naïve Bayes, SVM y MLP).

In [ ]:
def calcular_metricas(y_real, y_pred):
    return {
        "exactitud": accuracy_score(y_real, y_pred),
        "precision": precision_score(y_real, y_pred, average="macro", zero_division=0),
        "recall":    recall_score(y_real, y_pred, average="macro", zero_division=0),
        "f1":        f1_score(y_real, y_pred, average="macro", zero_division=0),
    }


def dividir_holdout(X, y):
    # Particion Hold Out 70/30 estratificada, identica para todo el equipo
    return train_test_split(X, y, test_size=TEST_SIZE, stratify=y, random_state=SEMILLA)


def obtener_kfold():
    return StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEMILLA)


def _entrenar_y_medir(modelo, X_tr, y_tr, X_te, y_te, escalar):
    modelo = copy.deepcopy(modelo)              # modelo nuevo en cada particion
    if escalar:
        escalador = StandardScaler().fit(X_tr)  # se ajusta SOLO con entrenamiento
        X_tr, X_te = escalador.transform(X_tr), escalador.transform(X_te)
    t0 = time.perf_counter(); modelo.fit(X_tr, y_tr)
    t1 = time.perf_counter(); y_pred = modelo.predict(X_te)
    t2 = time.perf_counter()
    res = calcular_metricas(y_te, y_pred)
    res["t_fit_ms"] = (t1 - t0) * 1000
    res["t_pred_ms"] = (t2 - t1) * 1000
    return res, y_pred


def evaluar_holdout(modelo, X, y, escalar=True):
    X, y = np.asarray(X, dtype=float), np.asarray(y)
    X_tr, X_te, y_tr, y_te = dividir_holdout(X, y)
    res, y_pred = _entrenar_y_medir(modelo, X_tr, y_tr, X_te, y_te, escalar)
    return res, (y_te, y_pred)


def evaluar_kfold(modelo, X, y, escalar=True):
    X, y = np.asarray(X, dtype=float), np.asarray(y)
    filas = []
    for tr, te in obtener_kfold().split(X, y):
        res, _ = _entrenar_y_medir(modelo, X[tr], y[tr], X[te], y[te], escalar)
        filas.append(res)
    df = pd.DataFrame(filas)
    resumen = df.mean().to_dict()
    resumen.update({f"{c}_std": df[c].std() for c in ["exactitud", "f1"]})
    return resumen, df

print("Funciones de evaluación listas")

# 2. Datasets

Para evaluar el comportamiento de los clasificadores se utilizan dos conjuntos de datos: uno sintético generado
mediante Scikit-learn y otro real correspondiente al conjunto Wine. Ambos cuentan con tres clases, lo que permite una
comparación consistente entre los clasificadores implementados.

El dataset sintético permite analizar el desempeño de los algoritmos bajo condiciones controladas, mientras que Wine
permite evaluar su comportamiento sobre datos reales con características numéricas de diferente escala.

## 2.1 Dataset sintético con ruido estructurado

Este dataset es el generado por el **Integrante 1**; se usa exactamente el mismo código para que los tres integrantes
trabajen con los mismos datos.

Se construye con `make_classification`: 1,000 muestras, seis características y tres clases. Para simular condiciones
de clasificación menos ideales se introduce un 5 % de etiquetas alteradas (`flip_y`) y se agrega **ruido
correlacionado**: el mismo vector de ruido se suma a las dos primeras características. Como ese ruido afecta a ambas
de la misma forma, no es aleatorio e independiente sino que tiene una estructura, y por eso se considera
**ruido estructurado**.

In [ ]:
# Generacion del dataset sintetico (codigo del Integrante 1)
X_sintetico, y_sintetico = make_classification(
    n_samples=1000,
    n_features=6,
    n_informative=4,
    n_redundant=2,
    n_classes=3,
    n_clusters_per_class=1,
    weights=[0.33, 0.33, 0.34],
    flip_y=0.05,
    class_sep=1.0,
    random_state=SEMILLA
)

X_sintetico_sin_ruido = X_sintetico.copy()   # copia solo para analizar el efecto del ruido

# Ruido correlacionado: el mismo vector se suma a las dos primeras caracteristicas
rng = np.random.default_rng(SEMILLA)
ruido = rng.normal(0, 0.5, size=len(X_sintetico))
X_sintetico[:, 0] += ruido
X_sintetico[:, 1] += ruido

# DataFrame para el analisis exploratorio
df_sintetico = pd.DataFrame(X_sintetico, columns=[f"x{i+1}" for i in range(X_sintetico.shape[1])])
df_sintetico["Clase"] = y_sintetico

print("Dataset sintético generado")
print("Muestras:", X_sintetico.shape[0])
print("Características:", X_sintetico.shape[1])
print("Clases:", np.unique(y_sintetico))
df_sintetico.head()

## 2.2 Parámetros del dataset sintético

| Parámetro | Valor | Explicación |
|---|---|---|
| `n_samples` | 1000 | Número de muestras generadas. |
| `n_features` | 6 | Número total de características por muestra. |
| `n_informative` | 4 | Características que aportan información útil para distinguir las clases. |
| `n_redundant` | 2 | Características que son combinaciones lineales de las informativas. |
| `n_classes` | 3 | Número de clases. |
| `n_clusters_per_class` | 1 | Cada clase se genera como un solo grupo de puntos. |
| `weights` | [0.33, 0.33, 0.34] | Proporción de muestras de cada clase (clases balanceadas). |
| `flip_y` | 0.05 | Aproximadamente 5 % de las etiquetas se asignan al azar (ruido en las etiquetas). |
| `class_sep` | 1.0 | Separación entre clases; valores menores producen más traslape. |
| `random_state` | 42 | Semilla que permite reproducir exactamente el mismo dataset. |
| Ruido correlacionado | $\mathcal{N}(0,\,0.5)$ | Un mismo vector de ruido sumado a `x1` y `x2` (ruido estructurado). |

## 2.3 Carga del dataset real: Wine

El segundo conjunto de datos es Wine, disponible en Scikit-learn y originalmente publicado en el repositorio de
aprendizaje automático de UCI. Contiene 178 muestras de tres clases de vino, descritas por 13 características
numéricas reales asociadas a propiedades químicas, como alcohol, ácido málico, cenizas, alcalinidad y magnesio.

Cumple el requisito de la práctica de un dataset benchmark con al menos 4 rasgos numéricos reales y 3 clases. Además,
sus características tienen escalas muy distintas, lo que lo hace adecuado para analizar la importancia de la
normalización en los algoritmos basados en distancias.

In [ ]:
wine = load_wine()
X_wine, y_wine = wine.data, wine.target

df_wine = pd.DataFrame(X_wine, columns=wine.feature_names)
df_wine["Clase"] = y_wine

print("Dataset Wine")
print("Muestras:", X_wine.shape[0])
print("Características:", X_wine.shape[1])
print("Clases:", np.unique(y_wine))
df_wine.head()

# 3. Análisis exploratorio de datos

Antes de implementar los clasificadores se realiza un análisis exploratorio de ambos conjuntos de datos con el fin
de:

- Verificar que los datos estén completos (sin valores nulos ni duplicados).
- Conocer la distribución de las clases.
- Comparar las escalas de las características.
- Identificar qué características separan mejor las clases.
- Detectar correlaciones entre características y valores atípicos.

## 3.1 Funciones de análisis

Las funciones son genéricas: reciben cualquier `DataFrame` con una columna `Clase`, por lo que se aplican igual a los
dos datasets.

In [ ]:
def resumen_dataset(df, nombre, objetivo="Clase"):
    rasgos = df.drop(columns=objetivo)
    print(f"===== {nombre} =====")
    print(f"Muestras: {df.shape[0]}  |  Características: {rasgos.shape[1]}  |  Clases: {df[objetivo].nunique()}")
    print(f"Valores nulos: {int(df.isnull().sum().sum())}  |  Filas duplicadas: {int(df.duplicated().sum())}")
    conteo = df[objetivo].value_counts().sort_index()
    print("\nDistribución de clases:")
    print(pd.DataFrame({"muestras": conteo, "%": (100 * conteo / len(df)).round(1)}))
    return rasgos.describe().T.round(3)


def graficar_clases(df, nombre, objetivo="Clase"):
    plt.figure(figsize=(5, 3.5))
    sns.countplot(data=df, x=objetivo, color="#4C72B0")
    plt.title(f"Distribución de clases — {nombre}")
    plt.xlabel("Clase"); plt.ylabel("Número de muestras")
    plt.tight_layout(); plt.show()


def graficar_histogramas(df, nombre, objetivo="Clase", columnas=4):
    rasgos = [c for c in df.columns if c != objetivo]
    filas = int(np.ceil(len(rasgos) / columnas))
    fig, axes = plt.subplots(filas, columnas, figsize=(4 * columnas, 3 * filas))
    axes = np.atleast_1d(axes).ravel()
    for ax, r in zip(axes, rasgos):
        sns.histplot(data=df, x=r, hue=objetivo, element="step", stat="density",
                     common_norm=False, palette="Set2", ax=ax, legend=(r == rasgos[0]))
        ax.set_title(r, fontsize=10); ax.set_xlabel("")
    for ax in axes[len(rasgos):]:
        ax.remove()
    fig.suptitle(f"Distribución de cada característica por clase — {nombre}", y=1.01)
    plt.tight_layout(); plt.show()


def graficar_boxplots(df, nombre, objetivo="Clase", titulo_extra=""):
    plt.figure(figsize=(max(8, 0.8 * (df.shape[1] - 1)), 4))
    sns.boxplot(data=df.drop(columns=objetivo), color="#8DA0CB")
    plt.xticks(rotation=45, ha="right")
    plt.title(f"Boxplot de características — {nombre} {titulo_extra}")
    plt.tight_layout(); plt.show()


def graficar_correlacion(df, nombre, objetivo="Clase"):
    corr = df.drop(columns=objetivo).corr()
    n = corr.shape[0]
    plt.figure(figsize=(0.65 * n + 3, 0.55 * n + 2))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1,
                square=True, annot_kws={"size": 7})
    plt.title(f"Matriz de correlación — {nombre}")
    plt.tight_layout(); plt.show()
    pares = (corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack()
                 .sort_values(key=abs, ascending=False))
    print("Pares de características más correlacionados (r):")
    print(pares.head(5).round(3).to_string())


def contar_outliers_iqr(df, objetivo="Clase"):
    # Valores fuera de [Q1 - 1.5*IQR, Q3 + 1.5*IQR]
    rasgos = df.drop(columns=objetivo)
    q1, q3 = rasgos.quantile(0.25), rasgos.quantile(0.75)
    iqr = q3 - q1
    fuera = (rasgos < q1 - 1.5 * iqr) | (rasgos > q3 + 1.5 * iqr)
    tabla = pd.DataFrame({"outliers": fuera.sum(), "%": (100 * fuera.mean()).round(1)})
    return tabla[tabla["outliers"] > 0].sort_values("outliers", ascending=False)


def medias_por_clase(df, objetivo="Clase"):
    return df.groupby(objetivo).mean().T.round(3)

## 3.2 Análisis exploratorio — Wine

In [ ]:
resumen_dataset(df_wine, "Wine")

In [ ]:
graficar_clases(df_wine, "Wine")

In [ ]:
graficar_histogramas(df_wine, "Wine")

In [ ]:
graficar_boxplots(df_wine, "Wine", titulo_extra="(escala original)")

In [ ]:
graficar_correlacion(df_wine, "Wine")

In [ ]:
contar_outliers_iqr(df_wine)

In [ ]:
# Vista conjunta de cuatro caracteristicas que separan bien las clases
sns.pairplot(df_wine[["alcohol", "flavanoids", "color_intensity", "proline", "Clase"]],
             hue="Clase", palette="Set2", corner=True, plot_kws={"s": 18})
plt.show()

### Observaciones — Wine

- **Datos completos:** no hay valores nulos ni filas duplicadas, por lo que no se requiere imputación.
- **Clases moderadamente desbalanceadas:** 59, 71 y 48 muestras (33.1 %, 39.9 % y 27.0 %). Por eso las particiones
  se hacen de forma **estratificada**, para conservar estas proporciones en entrenamiento y prueba.
- **Escalas muy diferentes:** `proline` tiene media cercana a 747 y desviación de 315, mientras que `hue` tiene media
  cercana a 0.96 y desviación de 0.23. Sin normalizar, `proline` y `magnesium` dominarían el cálculo de distancias en
  Distancia Mínima, 1-NN y K-NN.
- **Características muy correlacionadas:** `total_phenols` con `flavanoids` (r ≈ 0.86) y `flavanoids` con
  `od280/od315_of_diluted_wines` (r ≈ 0.79). Esto es relevante para Naïve Bayes, que supone independencia entre
  características.
- **Características más discriminantes:** en los histogramas y el pairplot, `flavanoids`, `proline`,
  `color_intensity` y `alcohol` muestran clases claramente separadas, por lo que se espera un buen desempeño general.
- **Pocos valores atípicos:** como máximo 2.2 % por característica (criterio IQR), así que no se eliminan muestras.

## 3.3 Análisis exploratorio — Sintético

In [ ]:
resumen_dataset(df_sintetico, "Sintético")

In [ ]:
graficar_clases(df_sintetico, "Sintético")

In [ ]:
graficar_histogramas(df_sintetico, "Sintético", columnas=3)

In [ ]:
graficar_boxplots(df_sintetico, "Sintético", titulo_extra="(escala original)")

In [ ]:
graficar_correlacion(df_sintetico, "Sintético")

In [ ]:
contar_outliers_iqr(df_sintetico)

In [ ]:
medias_por_clase(df_sintetico)

### Efecto del ruido correlacionado

Se compara el dataset antes y después de agregar el ruido para mostrar qué cambia en `x1` y `x2`.

In [ ]:
antes = pd.DataFrame(X_sintetico_sin_ruido[:, :2], columns=["x1", "x2"])
despues = df_sintetico[["x1", "x2"]]

print(f"Correlación x1-x2 antes del ruido:   {antes.corr().iloc[0, 1]: .3f}")
print(f"Correlación x1-x2 después del ruido: {despues.corr().iloc[0, 1]: .3f}")
print(f"Desviación de x1: {antes['x1'].std():.3f} -> {despues['x1'].std():.3f}")
print(f"Desviación de x2: {antes['x2'].std():.3f} -> {despues['x2'].std():.3f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
for ax, datos, titulo in [(axes[0], X_sintetico_sin_ruido, "Antes del ruido"),
                          (axes[1], X_sintetico, "Después del ruido correlacionado")]:
    sc = ax.scatter(datos[:, 0], datos[:, 1], c=y_sintetico, cmap="viridis", alpha=0.6, s=14)
    ax.set_title(titulo); ax.set_xlabel("x1"); ax.set_ylabel("x2")
fig.colorbar(sc, ax=axes, label="Clase")
plt.show()

### Observaciones — Sintético

- **Datos completos y balanceados:** 1,000 muestras sin nulos ni duplicados, con 325, 331 y 344 muestras por clase.
- **Escalas parecidas:** las desviaciones estándar van de 1.0 a 1.7 aproximadamente. A diferencia de Wine, aquí la
  normalización no cambia tanto las escalas relativas, pero se aplica igual para tratar ambos datasets con el mismo
  procedimiento.
- **Características redundantes:** hay correlaciones fuertes, como `x1`–`x4` (r ≈ 0.83), `x2`–`x6` (r ≈ −0.76) y
  `x1`–`x6` (r ≈ 0.73). Son consecuencia de las 2 características redundantes, que son combinaciones lineales de las
  informativas.
- **Características poco discriminantes:** en la tabla de medias por clase, `x3` y `x5` tienen prácticamente la misma
  media en las tres clases, por lo que, vistas por separado, aportan poca información. En cambio, `x1`, `x4`, `x2` y
  `x6` sí cambian de media entre clases.
- **Efecto del ruido estructurado:** como el mismo ruido se suma a `x1` y `x2`, les agrega un componente común que
  empuja su correlación hacia valores positivos: la correlación original de −0.36 se debilita a −0.24. La dispersión
  también aumenta (de 1.69 a 1.74 en `x1` y de 1.38 a 1.48 en `x2`), lo que produce más traslape entre clases en esas
  dos características. El efecto es moderado porque la desviación del ruido (0.5) es menor que la de las
  características (alrededor de 1.5).
- **Clases traslapadas:** junto con el 5 % de etiquetas alteradas, se espera que este dataset sea más difícil que Wine
  y que ningún clasificador alcance una exactitud cercana al 100 %.
- **Valores atípicos moderados:** hasta 4.6 % en `x4` según el criterio IQR. Son parte de la generación del dataset y
  no errores de captura, por lo que no se eliminan.

# 4. Normalización

Se utiliza **estandarización (z-score)**:

$$z = \frac{x - \mu}{\sigma}$$

donde $\mu$ y $\sigma$ son la media y la desviación estándar de cada característica.

**Regla importante:** $\mu$ y $\sigma$ se calculan **únicamente con el conjunto de entrenamiento** y después se aplican
esos mismos valores al conjunto de prueba. Si se calcularan con todo el dataset, el modelo tendría información del
conjunto de prueba (*data leakage*) y los resultados serían demasiado optimistas. En K-Fold la normalización se
repite dentro de cada fold, lo cual ya hacen las funciones de la sección 1.4.

**¿Por qué z-score y no Min-Max?** Min-Max depende del valor mínimo y máximo, así que un solo valor atípico comprime al
resto de los datos en un rango muy pequeño. Como ambos datasets tienen valores atípicos, el z-score es más estable.

**¿Qué clasificadores la necesitan?** Distancia Mínima, 1-NN y K-NN (se basan en distancias), SVM (se basa en
distancias y productos punto) y MLP (el entrenamiento por gradiente converge mejor con datos escalados). Naïve Bayes
Gaussiano modela cada característica por separado, por lo que no la necesita, pero tampoco le afecta.

In [ ]:
def normalizar(X_train, X_test):
    # Estandariza con la media y desviacion del ENTRENAMIENTO
    escalador = StandardScaler().fit(X_train)
    return escalador.transform(X_train), escalador.transform(X_test), escalador


# Particiones Hold Out 70/30 de ambos datasets
Xw_train, Xw_test, yw_train, yw_test = dividir_holdout(X_wine, y_wine)
Xs_train, Xs_test, ys_train, ys_test = dividir_holdout(X_sintetico, y_sintetico)

# Normalizacion
Xw_train_n, Xw_test_n, escalador_wine = normalizar(Xw_train, Xw_test)
Xs_train_n, Xs_test_n, escalador_sint = normalizar(Xs_train, Xs_test)

for nombre, Xtr, Xte, Xtr_n, Xte_n in [("Wine", Xw_train, Xw_test, Xw_train_n, Xw_test_n),
                                       ("Sintético", Xs_train, Xs_test, Xs_train_n, Xs_test_n)]:
    print(f"{nombre}: entrenamiento {Xtr.shape}, prueba {Xte.shape}")
    print(f"  Entrenamiento normalizado -> media {Xtr_n.mean():.3f}, desviación {Xtr_n.std():.3f}")
    print(f"  Prueba normalizada        -> media {Xte_n.mean():.3f}, desviación {Xte_n.std():.3f}\n")

El conjunto de entrenamiento queda con media 0 y desviación 1. El de prueba queda **cerca** de esos valores, pero no
exactamente, porque se transformó con los parámetros del entrenamiento. Ese es el comportamiento correcto.

In [ ]:
# Media y desviacion por caracteristica en Wine: antes y despues
comparacion = pd.DataFrame({
    "media_original": Xw_train.mean(axis=0),
    "desv_original": Xw_train.std(axis=0),
    "media_normalizada": Xw_train_n.mean(axis=0),
    "desv_normalizada": Xw_train_n.std(axis=0),
}, index=wine.feature_names)
comparacion.round(3)

In [ ]:
df_wine_n = pd.DataFrame(Xw_train_n, columns=wine.feature_names); df_wine_n["Clase"] = yw_train
df_sint_n = pd.DataFrame(Xs_train_n, columns=df_sintetico.columns[:-1]); df_sint_n["Clase"] = ys_train

graficar_boxplots(df_wine, "Wine", titulo_extra="(antes de normalizar)")
graficar_boxplots(df_wine_n, "Wine", titulo_extra="(entrenamiento normalizado)")
graficar_boxplots(df_sint_n, "Sintético", titulo_extra="(entrenamiento normalizado)")

# 5. Datasets preparados para el equipo

Se exporta cada dataset **sin normalizar** (la normalización se hace dentro de cada partición) con dos columnas
adicionales que fijan las particiones para todos los integrantes:

- `holdout`: indica si la muestra pertenece a `train` o `test` en el Hold Out 70/30.
- `fold`: número de fold (0 a 4) en el que la muestra forma parte del conjunto de prueba en K-Fold.

Estos archivos se suben al repositorio y sirven el 8 de noviembre para verificar que los tres integrantes usaron
exactamente las mismas divisiones.

In [ ]:
def exportar_con_particiones(df, archivo, objetivo="Clase"):
    X, y = df.drop(columns=objetivo).values, df[objetivo].values
    indices = np.arange(len(df))
    _, idx_test = train_test_split(indices, test_size=TEST_SIZE, stratify=y, random_state=SEMILLA)
    salida = df.copy()
    salida["holdout"] = "train"
    salida.loc[idx_test, "holdout"] = "test"
    salida["fold"] = -1
    for f, (_, te) in enumerate(obtener_kfold().split(X, y)):
        salida.loc[te, "fold"] = f
    salida.to_csv(archivo, index=False)
    print(f"{archivo}: {len(salida)} filas | Hold Out -> {salida['holdout'].value_counts().to_dict()} "
          f"| muestras por fold -> {salida['fold'].value_counts().sort_index().tolist()}")
    return salida

wine_preparado = exportar_con_particiones(df_wine, "wine_preparado.csv")
sintetico_preparado = exportar_con_particiones(df_sintetico, "sintetico_preparado.csv")

In [ ]:
# Descargar los archivos generados (solo funciona en Google Colab)
try:
    from google.colab import files
    for archivo in ["requirements.txt", "wine_preparado.csv", "sintetico_preparado.csv"]:
        files.download(archivo)
except ImportError:
    print("Fuera de Colab: los archivos quedaron guardados en la carpeta de trabajo.")

# 6. Implementación K-NN desde cero

El algoritmo K-Nearest Neighbors (K-NN), o K vecinos más cercanos, es un método de clasificación supervisada basado en
la proximidad entre las observaciones.

Su funcionamiento consiste en identificar los $K$ ejemplos del conjunto de entrenamiento que se encuentran más cerca de
una muestra desconocida. Posteriormente, se asigna a dicha muestra la clase que aparece con mayor frecuencia entre sus
vecinos.

En esta práctica el algoritmo se implementa desde cero, utilizando NumPy para calcular las distancias y realizar la
selección de los vecinos, sin recurrir al clasificador `KNeighborsClassifier` de Scikit-learn.

Para determinar la proximidad entre las muestras se utiliza por defecto la distancia euclidiana:

$$d(x, y) = \sqrt{\sum_{i=1}^{n} (x_i - y_i)^2}$$

donde $x$ e $y$ representan dos muestras y $n$ corresponde al número de características. La implementación también
permite usar la distancia Manhattan, que servirá para probar parámetros diferentes en la optimización:

$$d(x, y) = \sum_{i=1}^{n} |x_i - y_i|$$

El algoritmo se desarrolla en tres etapas:

1. **Entrenamiento:** almacenar las muestras y sus etiquetas. K-NN es un método *perezoso* (*lazy learning*): no
   construye ningún modelo, todo el trabajo ocurre al predecir.
2. **Distancias:** calcular la distancia entre cada muestra de prueba y **todas** las muestras de entrenamiento.
3. **Votación:** seleccionar los $K$ vecinos más cercanos y asignar la clase más frecuente entre ellos.
   - Votación `uniforme`: cada vecino vale un voto (comportamiento clásico).
   - Votación `distancia`: cada vecino vota con peso $1/d$, de modo que los más cercanos pesan más.
   - **Desempate:** si dos o más clases empatan en votos, gana la del vecino más cercano entre las empatadas.

Los parámetros por defecto (`k=5`, euclidiana, votación uniforme) son los mismos que usa `KNeighborsClassifier` de
Scikit-learn, para poder comparar después "parámetros por defecto contra parámetros modificados".

## 6.1 Implementación

La clase hereda de `ClassifierMixin` y `BaseEstimator` de Scikit-learn **solo** para que sea compatible con
herramientas como `Pipeline`, `cross_val_score` y `GridSearchCV` (igual que la clase `DistanciaMinima` del
Integrante 1). Todo el algoritmo, distancias, selección de vecinos y votación, está programado con NumPy.

In [ ]:
class KNNDesdeCero(ClassifierMixin, BaseEstimator):
    # Clasificador K vecinos mas cercanos implementado desde cero con NumPy

    def __init__(self, k=5, metrica="euclidiana", pesos="uniforme"):
        self.k = k
        self.metrica = metrica
        self.pesos = pesos

    def _validar_parametros(self):
        if not (isinstance(self.k, (int, np.integer)) and self.k >= 1):
            raise ValueError("k debe ser un entero mayor o igual a 1")
        if self.metrica not in ("euclidiana", "manhattan"):
            raise ValueError("metrica debe ser 'euclidiana' o 'manhattan'")
        if self.pesos not in ("uniforme", "distancia"):
            raise ValueError("pesos debe ser 'uniforme' o 'distancia'")

    # ---------- 1. Entrenamiento: solo memoriza los datos ----------
    def fit(self, X, y):
        self._validar_parametros()
        self.X_train_ = np.asarray(X, dtype=float)
        self.y_train_ = np.asarray(y)
        self.classes_ = np.unique(self.y_train_)
        return self

    # ---------- 2. Matriz de distancias (n_prueba x n_entrenamiento) ----------
    def _distancias(self, X):
        if self.metrica == "euclidiana":
            # ||a - b||^2 = ||a||^2 + ||b||^2 - 2 a.b  -> calculo vectorizado, sin bucles
            d2 = (np.sum(X ** 2, axis=1)[:, np.newaxis]
                  + np.sum(self.X_train_ ** 2, axis=1)[np.newaxis, :]
                  - 2 * X @ self.X_train_.T)
            return np.sqrt(np.maximum(d2, 0))     # maximum evita negativos por redondeo
        # Manhattan: se calcula por bloques de 256 muestras para no saturar la memoria
        bloques = [np.abs(X[i:i + 256, np.newaxis, :] - self.X_train_[np.newaxis, :, :]).sum(axis=2)
                   for i in range(0, len(X), 256)]
        return np.vstack(bloques)

    # ---------- 3a. K vecinos ordenados del mas cercano al mas lejano ----------
    def kvecinos(self, X):
        X = np.asarray(X, dtype=float)
        D = self._distancias(X)
        k = min(self.k, D.shape[1])
        idx = np.argpartition(D, k - 1, axis=1)[:, :k]          # los k menores en tiempo lineal
        dist = np.take_along_axis(D, idx, axis=1)
        orden = np.argsort(dist, axis=1, kind="stable")          # solo se ordenan esos k
        return np.take_along_axis(idx, orden, axis=1), np.take_along_axis(dist, orden, axis=1)

    # ---------- 3b. Prediccion por votacion ----------
    def predict(self, X):
        idx, dist = self.kvecinos(X)
        etiquetas = self.y_train_[idx]
        w = 1.0 / (dist + 1e-12) if self.pesos == "distancia" else np.ones_like(dist)

        predicciones = np.empty(len(etiquetas), dtype=self.y_train_.dtype)
        for i in range(len(etiquetas)):
            votos = {}
            for clase, peso in zip(etiquetas[i], w[i]):
                votos[clase] = votos.get(clase, 0.0) + peso
            maximo = max(votos.values())
            empatadas = {c for c, v in votos.items() if np.isclose(v, maximo)}
            for clase in etiquetas[i]:           # vecinos ordenados por cercania
                if clase in empatadas:
                    predicciones[i] = clase
                    break
        return predicciones

print(KNNDesdeCero())

## 6.2 Prueba de escritorio

Ejemplo pequeño que se puede comprobar a mano: dos grupos de puntos en el plano y tres muestras nuevas. La tercera
(3, 3) queda entre los dos grupos, pero sus 3 vecinos más cercanos pertenecen al grupo A.

In [ ]:
X_juguete = np.array([[1, 1], [1, 2], [2, 1],      # clase 0 (grupo A)
                      [6, 6], [6, 7], [7, 6]])     # clase 1 (grupo B)
y_juguete = np.array([0, 0, 0, 1, 1, 1])
puntos_nuevos = np.array([[1.5, 1.5], [6.5, 6.5], [3, 3]])

knn_juguete = KNNDesdeCero(k=3).fit(X_juguete, y_juguete)
indices, distancias = knn_juguete.kvecinos(puntos_nuevos)
predicciones = knn_juguete.predict(puntos_nuevos)

for p, i, d, c in zip(puntos_nuevos, indices, distancias, predicciones):
    print(f"Punto {p} -> vecinos {i.tolist()} | distancias {np.round(d, 2).tolist()} | clase {c}")

plt.figure(figsize=(5, 5))
plt.scatter(X_juguete[:, 0], X_juguete[:, 1], c=y_juguete, cmap="coolwarm", s=90, edgecolor="k", label="Entrenamiento")
plt.scatter(puntos_nuevos[:, 0], puntos_nuevos[:, 1], c=predicciones, cmap="coolwarm", marker="*", s=300,
            edgecolor="k", label="Nuevos (color = clase predicha)")
plt.title("Prueba de escritorio K-NN (k = 3)"); plt.legend(loc="upper left"); plt.show()

Para el punto (3, 3), la distancia a (1, 2) y a (2, 1) es $\sqrt{2^2 + 1^2} = \sqrt{5} \approx 2.24$ y a (1, 1) es
$\sqrt{2^2 + 2^2} = \sqrt{8} \approx 2.83$; las tres muestras son de la clase 0, por lo que se predice la clase 0.
Coincide con el resultado del código.

## 6.3 Validación contra Scikit-learn

Para comprobar que la implementación es correcta, se compara contra `KNeighborsClassifier` con los mismos parámetros
y la misma partición Hold Out normalizada. Si las predicciones coinciden, el K-NN propio funciona correctamente.

`KNeighborsClassifier` **no** forma parte de la solución; solo se usa como referencia para validar.

In [ ]:
datasets = {"Wine": (X_wine, y_wine), "Sintético": (X_sintetico, y_sintetico)}

filas = []
for nombre, (X, y) in datasets.items():
    X_tr, X_te, y_tr, y_te = dividir_holdout(X, y)
    X_tr, X_te, _ = normalizar(X_tr, X_te)
    for k in [1, 3, 5, 7, 9]:
        for metrica, metrica_sk in [("euclidiana", "euclidean"), ("manhattan", "manhattan")]:
            pred_propio = KNNDesdeCero(k=k, metrica=metrica).fit(X_tr, y_tr).predict(X_te)
            pred_sk = KNeighborsClassifier(n_neighbors=k, metric=metrica_sk).fit(X_tr, y_tr).predict(X_te)
            filas.append({"dataset": nombre, "k": k, "métrica": metrica,
                          "predicciones_iguales_%": 100 * np.mean(pred_propio == pred_sk),
                          "exactitud_propio": accuracy_score(y_te, pred_propio),
                          "exactitud_sklearn": accuracy_score(y_te, pred_sk)})

validacion_knn = pd.DataFrame(filas)
print(f"Coincidencia promedio con Scikit-learn: {validacion_knn['predicciones_iguales_%'].mean():.2f} %")
validacion_knn.round(4)

**Resultado de la validación:**
- En **Wine** las predicciones coinciden al **100 %** en todas las combinaciones de $K$ y métrica.
- En el **sintético** coinciden entre 97.7 % y 100 %. Se revisaron una por una las muestras con predicción distinta
  y **todas son empates de votos** (por ejemplo, con $K = 3$, un vecino de cada clase). Scikit-learn desempata
  eligiendo la etiqueta de número menor; esta implementación elige la clase del vecino más cercano, que es un criterio
  más razonable. Con $K = 1$ no puede haber empates, y la coincidencia es del 100 % en ambos datasets.

Por lo tanto, la implementación desde cero es correcta.

## 6.4 Verificación de funcionamiento en ambos datasets

Prueba rápida con los parámetros por defecto (`k=5`, euclidiana, votación uniforme) en el Hold Out 70/30, usando las
funciones compartidas del equipo. La evaluación completa con Hold Out y K-Fold corresponde al 7 y 8 de noviembre.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, (nombre, (X, y)) in zip(axes, datasets.items()):
    resultado, (y_real, y_pred) = evaluar_holdout(KNNDesdeCero(k=5), X, y)
    print(f"{nombre:<10} exactitud {resultado['exactitud']:.4f} | F1 {resultado['f1']:.4f} | "
          f"entrenamiento {resultado['t_fit_ms']:.3f} ms | predicción {resultado['t_pred_ms']:.3f} ms")
    ConfusionMatrixDisplay(confusion_matrix(y_real, y_pred)).plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(f"K-NN desde cero (k=5) — {nombre}")
plt.tight_layout(); plt.show()

## 6.5 Efecto de la normalización en K-NN

Misma partición y mismo $K$, con y sin estandarización, evaluado con K-Fold (5 folds). Este resultado comprueba la
decisión tomada en la sección 4.

In [ ]:
filas = []
for nombre, (X, y) in datasets.items():
    sin_norm, _ = evaluar_kfold(KNNDesdeCero(k=5), X, y, escalar=False)
    con_norm, _ = evaluar_kfold(KNNDesdeCero(k=5), X, y, escalar=True)
    filas.append({"dataset": nombre,
                  "exactitud_sin_normalizar": sin_norm["exactitud"],
                  "exactitud_normalizado": con_norm["exactitud"],
                  "diferencia_puntos": 100 * (con_norm["exactitud"] - sin_norm["exactitud"])})
pd.DataFrame(filas).round(4)

**Resultado:** en **Wine** normalizar sube la exactitud de K-NN de 67.4 % a 97.2 % (casi 30 puntos), porque sin
normalizar `proline` (valores de cientos) y `magnesium` dominan la distancia y las demás características casi no
cuentan. En el **sintético** la diferencia es de solo 0.1 puntos, porque, como se vio en el análisis exploratorio, sus
características ya tienen escalas parecidas. Esto confirma que la normalización es indispensable para K-NN cuando las
escalas son distintas.

# 7. Implementación Naïve Bayes

Naïve Bayes es un clasificador probabilístico basado en el **teorema de Bayes**. Para una muestra
$x = (x_1, \dots, x_n)$ calcula la probabilidad de que pertenezca a cada clase $c$:

$$P(c \mid x) = \frac{P(c)\, P(x \mid c)}{P(x)}$$

- $P(c)$: probabilidad *a priori* de la clase (proporción de muestras de esa clase en el entrenamiento).
- $P(x \mid c)$: verosimilitud, qué tan probable es observar $x$ si la muestra es de la clase $c$.
- $P(x)$: es igual para todas las clases, por lo que no afecta la decisión.

La suposición "ingenua" (*naïve*) es que las características son **independientes entre sí dada la clase**, lo que
permite escribir la verosimilitud como un producto:

$$P(x \mid c) = \prod_{j=1}^{n} P(x_j \mid c)$$

Como las características de ambos datasets son numéricas continuas, se usa **Naïve Bayes Gaussiano**: cada
característica se modela con una distribución normal con media $\mu_{cj}$ y varianza $\sigma^2_{cj}$ calculadas por
clase:

$$P(x_j \mid c) = \frac{1}{\sqrt{2\pi\sigma^2_{cj}}} \exp\left(-\frac{(x_j - \mu_{cj})^2}{2\sigma^2_{cj}}\right)$$

La clase predicha es la de mayor probabilidad. Para evitar que el producto de muchas probabilidades pequeñas dé cero
por redondeo, se trabaja con logaritmos:

$$\hat{c} = \arg\max_c \left[ \log P(c) + \sum_{j=1}^{n} \log P(x_j \mid c) \right]$$

Se implementa con `GaussianNB` de Scikit-learn, con sus parámetros por defecto: `priors=None` (las probabilidades
a priori se estiman de los datos) y `var_smoothing=1e-9` (se suma una pequeña cantidad a las varianzas para evitar
divisiones entre cero).

## 7.1 Entrenamiento y parámetros aprendidos

Se entrena con el Hold Out 70/30 de Wine para ver qué aprende el modelo. A diferencia de K-NN, Naïve Bayes sí
construye un modelo: solo necesita guardar una probabilidad a priori por clase y una media y una varianza por cada
combinación de clase y característica.

In [ ]:
nb_wine = GaussianNB()
nb_wine.fit(Xw_train, yw_train)          # datos sin normalizar (ver seccion 7.3)

print("Probabilidades a priori P(c):", np.round(nb_wine.class_prior_, 4))
print("Muestras por clase en entrenamiento:", nb_wine.class_count_.astype(int))
print(f"Parámetros aprendidos: {nb_wine.theta_.size} medias + {nb_wine.var_.size} varianzas "
      f"+ {nb_wine.class_prior_.size} probabilidades a priori")

pd.DataFrame(nb_wine.theta_.T, index=wine.feature_names,
             columns=[f"media clase {c}" for c in nb_wine.classes_]).round(3)

## 7.2 Comprobación manual de una predicción

Para entender lo que hace `GaussianNB` internamente, se calcula a mano la probabilidad de cada clase para la muestra
de prueba en la que el modelo está menos seguro, usando las fórmulas anteriores y los parámetros aprendidos, y se compara con `predict_proba`.

In [ ]:
# Se elige la muestra de prueba en la que el modelo esta menos seguro, para que el ejemplo sea interesante
i = np.argmin(nb_wine.predict_proba(Xw_test).max(axis=1))
x = Xw_test[i]

log_prior = np.log(nb_wine.class_prior_)
log_verosimilitud = -0.5 * np.sum(np.log(2 * np.pi * nb_wine.var_)
                                  + (x - nb_wine.theta_) ** 2 / nb_wine.var_, axis=1)
log_posterior = log_prior + log_verosimilitud

# Normalizacion para que las probabilidades sumen 1 (equivale a dividir entre P(x))
prob_manual = np.exp(log_posterior - log_posterior.max())
prob_manual /= prob_manual.sum()

print("log P(c) + Σ log P(x_j|c):  ", np.round(log_posterior, 3))
print("Probabilidad manual:         ", np.round(prob_manual, 4))
print("Probabilidad GaussianNB:     ", np.round(nb_wine.predict_proba([x])[0], 4))
print(f"Clase predicha manual: {np.argmax(prob_manual)} | GaussianNB: {nb_wine.predict([x])[0]} "
      f"| Clase real: {yw_test[i]}")

## 7.3 Verificación de funcionamiento en ambos datasets

Prueba rápida con los parámetros por defecto en el Hold Out 70/30, con las funciones compartidas del equipo. También
se compara con y sin normalización: como Naïve Bayes modela cada característica por separado con su propia media y
varianza, cambiar la escala de una característica no cambia sus decisiones.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
filas = []
for ax, (nombre, (X, y)) in zip(axes, datasets.items()):
    res_sin, (y_real, y_pred) = evaluar_holdout(GaussianNB(), X, y, escalar=False)
    res_con, _ = evaluar_holdout(GaussianNB(), X, y, escalar=True)
    filas.append({"dataset": nombre, "exactitud_sin_normalizar": res_sin["exactitud"],
                  "exactitud_normalizado": res_con["exactitud"], "f1": res_sin["f1"],
                  "t_fit_ms": res_sin["t_fit_ms"], "t_pred_ms": res_sin["t_pred_ms"]})
    ConfusionMatrixDisplay(confusion_matrix(y_real, y_pred)).plot(ax=ax, cmap="Purples", colorbar=False)
    ax.set_title(f"Naïve Bayes Gaussiano — {nombre}")
plt.tight_layout(); plt.show()
pd.DataFrame(filas).round(4)

**Resultado:**
- La exactitud es **idéntica con y sin normalización** en ambos datasets, como se esperaba. Por eso Naïve Bayes se
  evalúa sin normalizar.
- En **Wine** Naïve Bayes clasifica correctamente las 54 muestras de prueba (100 %). Las clases de Wine tienen
  distribuciones aproximadamente normales y bien separadas en varias características (sección 3.2), justo lo que
  supone el modelo.
- En el **sintético** obtiene 75.7 %, menos que K-NN (82.3 %). Este dataset tiene características redundantes y muy
  correlacionadas y ruido correlacionado entre `x1` y `x2`, lo que rompe la suposición de independencia (ver 7.4).

## 7.4 Relación con el análisis exploratorio

En la sección 3 se encontraron características fuertemente correlacionadas en ambos datasets (por ejemplo
`total_phenols`–`flavanoids` en Wine con r ≈ 0.86 y `x1`–`x4` en el sintético con r ≈ 0.83). Esto **viola la
suposición de independencia** de Naïve Bayes: al tratar dos características correlacionadas como independientes, el
modelo cuenta dos veces casi la misma información.

Aun así, Naïve Bayes suele clasificar bien, porque para acertar no necesita estimar las probabilidades con precisión:
basta con que la clase correcta quede con la probabilidad más alta. Este punto se retomará en la comparación de
resultados y en las conclusiones.

## 7.5 Comparación rápida K-NN y Naïve Bayes

Resumen de la verificación de ambos modelos con sus parámetros por defecto en el Hold Out 70/30. Es solo un primer
vistazo; la comparación formal de los seis clasificadores se hará con los resultados de todo el equipo.

In [ ]:
filas = []
for nombre, (X, y) in datasets.items():
    for modelo_nombre, modelo, escalar in [("K-NN desde cero (k=5)", KNNDesdeCero(k=5), True),
                                          ("Naïve Bayes Gaussiano", GaussianNB(), False)]:
        res, _ = evaluar_holdout(modelo, X, y, escalar=escalar)
        filas.append({"dataset": nombre, "modelo": modelo_nombre, **res})

resumen_dia_6 = pd.DataFrame(filas)
resumen_dia_6.round(4)

Las columnas `t_fit_ms` y `t_pred_ms` muestran la diferencia de comportamiento entre los dos modelos: K-NN casi no
tarda en entrenar (solo guarda los datos) y concentra su tiempo en la predicción, porque compara cada muestra de
prueba contra todo el entrenamiento. Naïve Bayes, en cambio, invierte un poco de tiempo en calcular medias y varianzas
al entrenar, y luego predice muy rápido. Este análisis se formalizará en la sección de complejidad temporal.

---
**Pendiente para los siguientes días:**
- 7 de noviembre: probar K-NN y Naïve Bayes en ambos datasets.
- 8 de noviembre: Hold Out y K-Fold completos; verificar que los tres integrantes usen las mismas divisiones.
- 9 de noviembre: optimización de K-NN (valores de K, métrica y votación).
- 10 de noviembre: complejidad temporal.

# 8. Evaluación Hold Out

# 9. Evaluación K-Fold

# 10. Optimización de K-NN

# 11. Comparación de resultados

# 12. Complejidad temporal

# 13. Conclusiones